# QSAR Workflow 
SMILES -> Features -> ML model -> Prediction

In [2]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd

def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "NumHDonors": Descriptors.NumHDonors(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
    }

data = pd.DataFrame({
    "SMILES": ["CCO", "CCN", "CCC"],
    "Activity": [5.2, 6.8, 4.1]
})

desc_df = data["SMILES"].apply(compute_descriptors).apply(pd.Series)
X = desc_df
y = data["Activity"]


In [3]:
from rdkit.Chem import AllChem
import numpy as np

def morgan_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    return np.array(fp)

X = np.array([morgan_fp(s) for s in data["SMILES"]])
y = data["Activity"]


In [4]:
# Regression (predict continuous value)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200)
model.fit(X_train, y_train)

pred = model.predict(X_test)


In [5]:
# Classification (active vs inactive)
from sklearn.ensemble import RandomForestClassifier

y_class = (y > 6).astype(int)  # threshold example
model = RandomForestClassifier()
model.fit(X_train, y_class_train)


NameError: name 'y_class_train' is not defined

In [6]:
from sklearn.metrics import r2_score, mean_squared_error

r2 = r2_score(y_test, pred)
rmse = mean_squared_error(y_test, pred, squared=False)


/Users/jwang/miniconda3/envs/ml-env/lib/python3.10/site-packages/sklearn/metrics/_regression.py:1283: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


TypeError: got an unexpected keyword argument 'squared'

In [ ]:
new_smiles = ["CCCO", "CCCl"]
X_new = np.array([morgan_fp(s) for s in new_smiles])
predictions = model.predict(X_new)
